# Notebook 5 — Image Quality Improvement & Model Refinement

This notebook focuses on improving the visual quality of the generative models developed in the previous phases.

The earlier notebooks primarily used a small synthetic dataset and short training runs for architectural validation. As a result, the generated images were structurally valid but visually noisy and low quality.

In this notebook we:
- train models using real WikiArt images
- increase training duration
- compare VAE, DCGAN, WGAN-GP, and CycleGAN quality
- evaluate generated images using FID and Inception Score
- visually compare generated samples
- save improved checkpoints
- generate a higher-quality portfolio

The objective is to improve actual image quality rather than simply increasing exported image resolution.

## 1. Environment Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from torch.optim import Adam

from generative_art_studio import config
from generative_art_studio.config import DEVICE
from generative_art_studio.utils import set_seed, plot_image_grid

set_seed(42)

print("Repository:", REPO_ROOT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available - using CPU")

## 2. Training Configuration

The previous experiments used only a few epochs. This notebook uses longer training runs on real WikiArt data.

If training takes too long, reduce the epoch values before starting.

In [ ]:
SEED = 42
IMAGE_SIZE = 64
BATCH_SIZE = 32

VAE_EPOCHS = 10
GAN_EPOCHS = 10
DCGAN_EPOCHS = 10
WGAN_EPOCHS = 10
CYCLEGAN_EPOCHS = 10

VAE_LATENT_DIM = 128
GAN_LATENT_DIM = 64
DCGAN_LATENT_DIM = 100

LEARNING_RATE = 2e-4
BETAS = (0.5, 0.999)

CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
QUALITY_OUTPUT_DIR = REPO_ROOT / "outputs" / "quality"
PORTFOLIO_DIR = REPO_ROOT / "outputs" / "portfolio_quality"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PORTFOLIO_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Portfolio directory:", PORTFOLIO_DIR)

## 3. Load the Real WikiArt Dataset

The unconditional models use all available WikiArt images. CycleGAN later separates the content and style domains.

In [ ]:
from generative_art_studio.data import get_image_dataset, get_dataloader

wikiart_dataset = get_image_dataset(
    REPO_ROOT / "data" / "wikiart",
    image_size=IMAGE_SIZE,
)

full_dataloader = get_dataloader(
    wikiart_dataset,
    batch_size=BATCH_SIZE,
)

print("Dataset exists:", (REPO_ROOT / "data" / "wikiart").exists())
print("Dataset size:", len(wikiart_dataset))
print("Classes:", wikiart_dataset.classes)

real_images, real_labels = next(iter(full_dataloader))
print("Batch shape:", real_images.shape)
print("Label shape:", real_labels.shape)

In [ ]:
plot_image_grid(
    real_images[:16],
    nrow=4,
    title="Real WikiArt Training Images",
)

## 4. Create Separate CycleGAN Domains

In [ ]:
from torch.utils.data import Subset

content_indices = [
    i for i, label in enumerate(wikiart_dataset.targets)
    if label == wikiart_dataset.class_to_idx["content"]
]

style_indices = [
    i for i, label in enumerate(wikiart_dataset.targets)
    if label == wikiart_dataset.class_to_idx["style"]
]

content_dataset = Subset(wikiart_dataset, content_indices)
style_dataset = Subset(wikiart_dataset, style_indices)

content_dataloader = get_dataloader(content_dataset, batch_size=16)
style_dataloader = get_dataloader(style_dataset, batch_size=16)

print("Content images:", len(content_dataset))
print("Style images:", len(style_dataset))

# Part A — Improved VAE

In [ ]:
from generative_art_studio.models.autoencoders import VAE
from generative_art_studio.training.train_vae import train_vae

vae = VAE(
    in_channels=3,
    latent_dim=VAE_LATENT_DIM,
).to(DEVICE)

vae_optimizer = Adam(vae.parameters(), lr=1e-3)

vae_history = train_vae(
    vae,
    full_dataloader,
    vae_optimizer,
    device=DEVICE,
    epochs=VAE_EPOCHS,
    kl_weight=1.0,
)

print("VAE training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(vae_history.total, label="Total Loss")
plt.plot(vae_history.recon, label="Reconstruction Loss")
plt.plot(vae_history.kl, label="KL Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Improved VAE Training")
plt.legend()
plt.show()

In [ ]:
vae.eval()

with torch.no_grad():
    vae_recon, _, _ = vae(real_images.to(DEVICE))
    vae_samples = vae.sample(16, device=DEVICE)

plot_image_grid(real_images[:16], nrow=4, title="VAE — Real Images")
plot_image_grid(vae_recon[:16].cpu(), nrow=4, title="VAE — Improved Reconstructions")
plot_image_grid(vae_samples.cpu(), nrow=4, title="VAE — Improved Samples")

In [ ]:
vae_checkpoint = CHECKPOINT_DIR / "vae_quality.pt"
torch.save(vae.state_dict(), vae_checkpoint)
print("Saved:", vae_checkpoint)

# Part B — Improved Vanilla GAN

In [ ]:
from generative_art_studio.models.gans import VanillaGenerator, VanillaDiscriminator
from generative_art_studio.training.train_gan import train_gan

vanilla_g = VanillaGenerator(latent_dim=GAN_LATENT_DIM).to(DEVICE)
vanilla_d = VanillaDiscriminator().to(DEVICE)

vanilla_g_optimizer = Adam(vanilla_g.parameters(), lr=LEARNING_RATE, betas=BETAS)
vanilla_d_optimizer = Adam(vanilla_d.parameters(), lr=LEARNING_RATE, betas=BETAS)

vanilla_history = train_gan(
    vanilla_g,
    vanilla_d,
    full_dataloader,
    vanilla_g_optimizer,
    vanilla_d_optimizer,
    GAN_LATENT_DIM,
    device=DEVICE,
    epochs=GAN_EPOCHS,
)

print("Vanilla GAN training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(vanilla_history.g_loss, label="Generator Loss")
plt.plot(vanilla_history.d_loss, label="Discriminator Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Improved Vanilla GAN Training")
plt.legend()
plt.show()

vanilla_g.eval()
z = torch.randn(16, GAN_LATENT_DIM, device=DEVICE)
with torch.no_grad():
    vanilla_samples = vanilla_g(z)

plot_image_grid(vanilla_samples.cpu(), nrow=4, title="Improved Vanilla GAN Samples")

In [ ]:
vanilla_checkpoint = CHECKPOINT_DIR / "vanilla_gan_quality.pt"
torch.save(vanilla_g.state_dict(), vanilla_checkpoint)
print("Saved:", vanilla_checkpoint)

# Part C — Improved DCGAN

In [ ]:
from generative_art_studio.models.gans import (
    DCGANGenerator,
    DCGANDiscriminator,
    weights_init_dcgan,
)

dc_generator = DCGANGenerator(
    latent_dim=DCGAN_LATENT_DIM,
    feature_maps=32,
).to(DEVICE)

dc_discriminator = DCGANDiscriminator(
    feature_maps=32,
).to(DEVICE)

dc_generator.apply(weights_init_dcgan)
dc_discriminator.apply(weights_init_dcgan)

dc_g_optimizer = Adam(dc_generator.parameters(), lr=LEARNING_RATE, betas=BETAS)
dc_d_optimizer = Adam(dc_discriminator.parameters(), lr=LEARNING_RATE, betas=BETAS)

dc_history = train_gan(
    dc_generator,
    dc_discriminator,
    full_dataloader,
    dc_g_optimizer,
    dc_d_optimizer,
    DCGAN_LATENT_DIM,
    device=DEVICE,
    epochs=DCGAN_EPOCHS,
)

print("DCGAN training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(dc_history.g_loss, label="Generator Loss")
plt.plot(dc_history.d_loss, label="Discriminator Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Improved DCGAN Training")
plt.legend()
plt.show()

dc_generator.eval()
z = torch.randn(16, DCGAN_LATENT_DIM, device=DEVICE)
with torch.no_grad():
    dc_samples = dc_generator(z)

plot_image_grid(dc_samples.cpu(), nrow=4, title="Improved DCGAN Samples")

In [ ]:
dc_checkpoint = CHECKPOINT_DIR / "dcgan_quality.pt"
torch.save(dc_generator.state_dict(), dc_checkpoint)
print("Saved:", dc_checkpoint)

# Part D — Improved WGAN-GP

In [ ]:
from generative_art_studio.models.gans import WGANCritic
from generative_art_studio.training.losses import (
    wgan_critic_loss,
    wgan_generator_loss,
    gradient_penalty,
)

wgan_generator = DCGANGenerator(
    latent_dim=DCGAN_LATENT_DIM,
    feature_maps=32,
).to(DEVICE)

wgan_critic = WGANCritic(feature_maps=32).to(DEVICE)

wgan_g_optimizer = Adam(wgan_generator.parameters(), lr=LEARNING_RATE, betas=BETAS)
wgan_c_optimizer = Adam(wgan_critic.parameters(), lr=LEARNING_RATE, betas=BETAS)

n_critic = config.N_CRITIC
gp_lambda = config.WGAN_GP_LAMBDA

print("N_CRITIC:", n_critic)
print("Gradient penalty lambda:", gp_lambda)

In [ ]:
wgan_g_losses = []
wgan_c_losses = []
wgan_gp_losses = []

for epoch in range(WGAN_EPOCHS):
    data_iter = iter(full_dataloader)

    for _ in range(len(full_dataloader)):
        for _ in range(n_critic):
            try:
                real_batch, _ = next(data_iter)
            except StopIteration:
                data_iter = iter(full_dataloader)
                real_batch, _ = next(data_iter)

            real_batch = real_batch.to(DEVICE)
            batch_size = real_batch.size(0)

            wgan_c_optimizer.zero_grad()

            z = torch.randn(batch_size, DCGAN_LATENT_DIM, device=DEVICE)
            fake_batch = wgan_generator(z)

            real_score = wgan_critic(real_batch)
            fake_score = wgan_critic(fake_batch.detach())

            critic_loss = wgan_critic_loss(real_score, fake_score)

            gp = gradient_penalty(
                wgan_critic,
                real_batch,
                fake_batch.detach(),
                device=DEVICE,
            )

            total_critic_loss = critic_loss + gp_lambda * gp
            total_critic_loss.backward()
            wgan_c_optimizer.step()

            wgan_c_losses.append(total_critic_loss.item())
            wgan_gp_losses.append(gp.item())

        wgan_g_optimizer.zero_grad()

        z = torch.randn(batch_size, DCGAN_LATENT_DIM, device=DEVICE)
        fake_batch = wgan_generator(z)
        fake_score = wgan_critic(fake_batch)

        generator_loss = wgan_generator_loss(fake_score)
        generator_loss.backward()
        wgan_g_optimizer.step()

        wgan_g_losses.append(generator_loss.item())

    print(
        f"Epoch [{epoch + 1}/{WGAN_EPOCHS}] | "
        f"Critic: {wgan_c_losses[-1]:.4f} | "
        f"Generator: {wgan_g_losses[-1]:.4f} | "
        f"GP: {wgan_gp_losses[-1]:.4f}"
    )

print("WGAN-GP training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(wgan_c_losses, label="Critic Loss")
plt.plot(wgan_g_losses, label="Generator Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Improved WGAN-GP Training")
plt.legend()
plt.show()

wgan_generator.eval()
z = torch.randn(16, DCGAN_LATENT_DIM, device=DEVICE)

with torch.no_grad():
    wgan_samples = wgan_generator(z)

plot_image_grid(wgan_samples.cpu(), nrow=4, title="Improved WGAN-GP Samples")

In [ ]:
wgan_checkpoint = CHECKPOINT_DIR / "wgan_gp_quality.pt"
torch.save(wgan_generator.state_dict(), wgan_checkpoint)
print("Saved:", wgan_checkpoint)

# Part E — Improved CycleGAN Style Transfer

CycleGAN is the main model for artistic style transfer. It learns A→B and B→A mappings using adversarial, cycle-consistency, and identity losses.

In [ ]:
from generative_art_studio.models.advanced import CycleGANGenerator, PatchGANDiscriminator
from generative_art_studio.training.losses import (
    patchgan_discriminator_loss,
    cycle_consistency_loss,
    identity_loss,
)

g_a2b_quality = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
g_b2a_quality = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)

d_a_quality = PatchGANDiscriminator(in_channels=3).to(DEVICE)
d_b_quality = PatchGANDiscriminator(in_channels=3).to(DEVICE)

cycle_g_optimizer = Adam(
    list(g_a2b_quality.parameters()) + list(g_b2a_quality.parameters()),
    lr=2e-4,
    betas=(0.5, 0.999),
)

cycle_d_a_optimizer = Adam(d_a_quality.parameters(), lr=2e-4, betas=(0.5, 0.999))
cycle_d_b_optimizer = Adam(d_b_quality.parameters(), lr=2e-4, betas=(0.5, 0.999))

lambda_cycle = 10.0
lambda_identity = 5.0

In [ ]:
cycle_g_losses = []
cycle_d_a_losses = []
cycle_d_b_losses = []

for epoch in range(CYCLEGAN_EPOCHS):
    for (real_a, _), (real_b, _) in zip(content_dataloader, style_dataloader):
        real_a = real_a.to(DEVICE)
        real_b = real_b.to(DEVICE)

        fake_b = g_a2b_quality(real_a)
        fake_a = g_b2a_quality(real_b)

        reconstructed_a = g_b2a_quality(fake_b)
        reconstructed_b = g_a2b_quality(fake_a)

        identity_b = g_a2b_quality(real_b)
        identity_a = g_b2a_quality(real_a)

        # Generator update
        cycle_g_optimizer.zero_grad()

        fake_b_pred = d_b_quality(fake_b)
        fake_a_pred = d_a_quality(fake_a)

        adv_a2b = F.binary_cross_entropy_with_logits(
            fake_b_pred, torch.ones_like(fake_b_pred)
        )
        adv_b2a = F.binary_cross_entropy_with_logits(
            fake_a_pred, torch.ones_like(fake_a_pred)
        )

        cycle_a = cycle_consistency_loss(real_a, reconstructed_a)
        cycle_b = cycle_consistency_loss(real_b, reconstructed_b)

        identity_a_loss = identity_loss(real_a, identity_a)
        identity_b_loss = identity_loss(real_b, identity_b)

        total_g_loss = (
            adv_a2b
            + adv_b2a
            + lambda_cycle * (cycle_a + cycle_b)
            + lambda_identity * (identity_a_loss + identity_b_loss)
        )

        total_g_loss.backward()
        cycle_g_optimizer.step()

        # Discriminator A
        cycle_d_a_optimizer.zero_grad()

        real_a_pred = d_a_quality(real_a)
        fake_a_pred = d_a_quality(fake_a.detach())

        d_a_loss = patchgan_discriminator_loss(real_a_pred, fake_a_pred)
        d_a_loss.backward()
        cycle_d_a_optimizer.step()

        # Discriminator B
        cycle_d_b_optimizer.zero_grad()

        real_b_pred = d_b_quality(real_b)
        fake_b_pred = d_b_quality(fake_b.detach())

        d_b_loss = patchgan_discriminator_loss(real_b_pred, fake_b_pred)
        d_b_loss.backward()
        cycle_d_b_optimizer.step()

        cycle_g_losses.append(total_g_loss.item())
        cycle_d_a_losses.append(d_a_loss.item())
        cycle_d_b_losses.append(d_b_loss.item())

    print(
        f"Epoch [{epoch + 1}/{CYCLEGAN_EPOCHS}] | "
        f"G: {cycle_g_losses[-1]:.4f} | "
        f"D_A: {cycle_d_a_losses[-1]:.4f} | "
        f"D_B: {cycle_d_b_losses[-1]:.4f}"
    )

print("Improved CycleGAN training complete.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cycle_g_losses, label="Generator Loss")
plt.plot(cycle_d_a_losses, label="Discriminator A Loss")
plt.plot(cycle_d_b_losses, label="Discriminator B Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Improved CycleGAN Training")
plt.legend()
plt.show()

In [ ]:
g_a2b_quality.eval()
g_b2a_quality.eval()

content_batch, _ = next(iter(content_dataloader))
content_batch = content_batch.to(DEVICE)

with torch.no_grad():
    stylized_batch = g_a2b_quality(content_batch)
    reconstructed_content = g_b2a_quality(stylized_batch)

plot_image_grid(content_batch[:8].cpu(), nrow=4, title="Original Content")
plot_image_grid(stylized_batch[:8].cpu(), nrow=4, title="Improved CycleGAN — Content to Style")
plot_image_grid(reconstructed_content[:8].cpu(), nrow=4, title="CycleGAN — Reconstructed Content")

In [ ]:
cycle_a2b_checkpoint = CHECKPOINT_DIR / "cyclegan_a2b_quality.pt"
cycle_b2a_checkpoint = CHECKPOINT_DIR / "cyclegan_b2a_quality.pt"

torch.save(g_a2b_quality.state_dict(), cycle_a2b_checkpoint)
torch.save(g_b2a_quality.state_dict(), cycle_b2a_checkpoint)

print("Saved:", cycle_a2b_checkpoint)
print("Saved:", cycle_b2a_checkpoint)

# Part F — FID and Inception Score

- Lower FID is better.
- Higher Inception Score is generally better.
- CycleGAN A→B must be compared with real style-domain images.
- Metrics should be interpreted together with visual inspection.

In [ ]:
from generative_art_studio.evaluation.metrics import (
    get_inception_feature_extractor,
    compute_fid,
    compute_inception_score,
)

extractor = get_inception_feature_extractor().to(DEVICE)
extractor.eval()

print("Inception feature extractor loaded.")

In [ ]:
def collect_images(dataloader, max_batches=10):
    batches = []

    for batch_index, (images, _) in enumerate(dataloader):
        batches.append(images)

        if batch_index + 1 >= max_batches:
            break

    return torch.cat(batches, dim=0)

real_eval = collect_images(full_dataloader, max_batches=10)
content_eval = collect_images(content_dataloader, max_batches=10)
style_eval = collect_images(style_dataloader, max_batches=10)

print("Full evaluation images:", len(real_eval))
print("Content evaluation images:", len(content_eval))
print("Style evaluation images:", len(style_eval))

In [ ]:
vae.eval()
dc_generator.eval()
wgan_generator.eval()
g_a2b_quality.eval()

with torch.no_grad():
    vae_eval = vae.sample(len(real_eval), device=DEVICE)

    z_dc = torch.randn(len(real_eval), DCGAN_LATENT_DIM, device=DEVICE)
    dc_eval = dc_generator(z_dc)

    z_wgan = torch.randn(len(real_eval), DCGAN_LATENT_DIM, device=DEVICE)
    wgan_eval = wgan_generator(z_wgan)

    cycle_eval = g_a2b_quality(content_eval.to(DEVICE))

print("Generated evaluation batches.")

In [ ]:
def evaluate_generated_images(real_images, generated_images, model_name):
    with torch.no_grad():
        real_features, _ = extractor(real_images.to(DEVICE))
        fake_features, fake_probs = extractor(generated_images.to(DEVICE))

    fid = compute_fid(
        real_features.cpu().numpy(),
        fake_features.cpu().numpy(),
    )

    is_mean, is_std = compute_inception_score(
        fake_probs.cpu().numpy()
    )

    result = {
        "Model": model_name,
        "FID": float(fid),
        "IS Mean": float(is_mean),
        "IS Std": float(is_std),
    }

    print(f"\n{model_name}")
    print(f"FID: {fid:.4f}")
    print(f"Inception Score: {is_mean:.4f} ± {is_std:.4f}")

    return result

results = []

results.append(evaluate_generated_images(real_eval, vae_eval, "VAE"))
results.append(evaluate_generated_images(real_eval, dc_eval, "DCGAN"))
results.append(evaluate_generated_images(real_eval, wgan_eval, "WGAN-GP"))

# CycleGAN A→B produces style-domain images, so compare against style_eval.
results.append(evaluate_generated_images(style_eval, cycle_eval, "CycleGAN A→B"))

quality_results = pd.DataFrame(results)
quality_results

## 10. Visual Quality Comparison

Inspect the samples using:
- sharpness
- structure
- color
- texture
- diversity
- artifacts
- artistic coherence

In [ ]:
plot_image_grid(vae_eval[:16].cpu(), nrow=4, title="VAE — Quality Samples")
plot_image_grid(dc_eval[:16].cpu(), nrow=4, title="DCGAN — Quality Samples")
plot_image_grid(wgan_eval[:16].cpu(), nrow=4, title="WGAN-GP — Quality Samples")
plot_image_grid(cycle_eval[:16].cpu(), nrow=4, title="CycleGAN — Style Transfer Samples")

# Part G — VAE Latent Interpolation

In [ ]:
from generative_art_studio.utils.latent_space import interpolate_latent

vae.eval()

z1 = torch.randn(VAE_LATENT_DIM, device=DEVICE)
z2 = torch.randn(VAE_LATENT_DIM, device=DEVICE)

latent_path = interpolate_latent(z1, z2, steps=8)

with torch.no_grad():
    interpolation_images = vae.decode(latent_path)

plot_image_grid(
    interpolation_images.cpu(),
    nrow=8,
    title="Improved VAE Latent Interpolation",
)

# Part H — Generate the Improved Portfolio

Upscaling only changes presentation size. The actual quality improvement comes from training on real data for longer.

In [ ]:
from generative_art_studio.app.model_registry import export_image

def export_generated_batch(images, output_dir, prefix, scale_factor=4):
    output_dir.mkdir(parents=True, exist_ok=True)
    exported = []

    for index, image in enumerate(images):
        output_path = output_dir / f"{prefix}_{index:03d}.png"

        export_image(
            image.detach().cpu(),
            output_path,
            scale_factor=scale_factor,
        )

        exported.append(output_path)

    return exported

In [ ]:
vae.eval()

with torch.no_grad():
    vae_portfolio = vae.sample(25, device=DEVICE)

vae_files = export_generated_batch(
    vae_portfolio,
    PORTFOLIO_DIR / "vae",
    "vae",
)

print("VAE images exported:", len(vae_files))

In [ ]:
dc_generator.eval()

z = torch.randn(25, DCGAN_LATENT_DIM, device=DEVICE)

with torch.no_grad():
    dc_portfolio = dc_generator(z)

dc_files = export_generated_batch(
    dc_portfolio,
    PORTFOLIO_DIR / "dcgan",
    "dcgan",
)

print("DCGAN images exported:", len(dc_files))

In [ ]:
wgan_generator.eval()

z = torch.randn(25, DCGAN_LATENT_DIM, device=DEVICE)

with torch.no_grad():
    wgan_portfolio = wgan_generator(z)

wgan_files = export_generated_batch(
    wgan_portfolio,
    PORTFOLIO_DIR / "wgan_gp",
    "wgan",
)

print("WGAN-GP images exported:", len(wgan_files))

In [ ]:
g_a2b_quality.eval()

content_portfolio = collect_images(
    content_dataloader,
    max_batches=4,
)

with torch.no_grad():
    cycle_portfolio = g_a2b_quality(
        content_portfolio.to(DEVICE)
    )

cycle_files = export_generated_batch(
    cycle_portfolio,
    PORTFOLIO_DIR / "cyclegan",
    "cyclegan",
)

print("CycleGAN images exported:", len(cycle_files))

# Part I — Portfolio Summary

In [ ]:
all_portfolio_files = list(PORTFOLIO_DIR.rglob("*.png"))

print("Total portfolio images:", len(all_portfolio_files))

for folder in sorted({file.parent for file in all_portfolio_files}):
    count = len(list(folder.glob("*.png")))
    print(f"{folder.name}: {count}")

# Part J — Final Reflection

## 1. Did image quality improve?

Compare the generated images from this notebook with the outputs from the earlier notebooks. Focus on structure, sharpness, color, texture, diversity, and artifacts.

## 2. Which model produced the best images?

Use a combination of visual quality, FID, Inception Score, diversity, and artistic coherence.

## 3. Did FID and visual judgment agree?

Explain whether the model with the lowest FID also appeared best visually. If they disagree, note that FID measures feature-distribution similarity rather than artistic quality.

## 4. Which model should be used for the final portfolio?

For independent generation, compare VAE, DCGAN, and WGAN-GP. For style transfer, CycleGAN is the primary candidate.

## 5. Limitations

The models operate at 64×64 resolution. More training, larger models, higher-resolution data, stronger architectures, and larger datasets could further improve quality.

# Final Conclusion

This notebook improves the generative-art pipeline by moving from short synthetic-data experiments to longer training runs using real WikiArt images.

The experiment evaluates VAE, DCGAN, WGAN-GP, and CycleGAN using quantitative metrics and visual inspection. Trained checkpoints are saved under `checkpoints/`, and higher-resolution portfolio images are exported under `outputs/portfolio_quality/`.

The resulting checkpoints can be integrated into the Phase 4 Streamlit platform after visual quality has been validated.